# Five advanced baseline models for CE-THGT

This Colab notebook trains and evaluates five research baselines on the chronological MOOCCubeX splits:

1. **NARM** — GRU plus local/global attention for session-aware recommendation.
2. **SASRec** — causal self-attention over the learner's ordered history.
3. **BERT4Rec** — bidirectional Transformer with a masked next-item position.
4. **LightGCN** — collaborative user–video graph propagation without feature transformations.
5. **HGT** — typed attention over user, video, concept, and course nodes.

All models use the same item vocabulary, chronological validation/test protocol, full-catalog ranking, and Recall@K, NDCG@K, and MRR@K metrics. Checkpoints and results are saved under `/content/drive/MyDrive/DataCon/baselines`.

> Run the preprocessing notebook first. This notebook expects `/content/drive/MyDrive/DataCon/processed`.

In [ ]:
# Colab setup: mount Drive and install only the required packages.
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pyarrow pandas tqdm torch-geometric

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict
import gc, json, math, os, random, time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from torch_geometric.data import HeteroData
    from torch_geometric.nn import HGTConv
except Exception as e:
    raise RuntimeError('Restart the runtime after installing torch-geometric, then run again.') from e

@dataclass
class Config:
    root: str = '/content/drive/MyDrive/DataCon'
    seed: int = 42
    max_len: int = 50
    hidden_dim: int = 128
    heads: int = 4
    layers: int = 2
    dropout: float = 0.1
    batch_size: int = 256
    eval_batch_size: int = 128
    epochs: int = 10
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    graph_learning_rate: float = 2e-3
    ks: tuple = (5, 10, 20)
    patience: int = 3
    num_workers: int = 2

CFG = Config()
ROOT = Path(CFG.root)
PROCESSED = ROOT/'processed'
SPLITS = PROCESSED/'splits'
GRAPH = PROCESSED/'graph'
OUT = ROOT/'baselines'
CHECKPOINTS = OUT/'checkpoints'
for p in [OUT, CHECKPOINTS]: p.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Output:', OUT)

## 1. Load chronological splits and construct a common vocabulary

Item index `0` is reserved for padding. Real videos use indices `1..N`. BERT4Rec uses `N+1` as its mask token. Validation histories contain training events; test histories additionally contain the validation target.

In [ ]:
required = [SPLITS/'train.parquet', SPLITS/'valid.parquet', SPLITS/'test.parquet']
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError(f'Run preprocessing first. Missing: {missing}')

train_df = pd.read_parquet(SPLITS/'train.parquet').sort_values(['user_id','timestamp'])
valid_df = pd.read_parquet(SPLITS/'valid.parquet').sort_values(['user_id','timestamp'])
test_df = pd.read_parquet(SPLITS/'test.parquet').sort_values(['user_id','timestamp'])

for frame in [train_df, valid_df, test_df]:
    frame['user_id'] = frame['user_id'].astype(str)
    frame['video_id'] = frame['video_id'].astype(str)
    frame['timestamp'] = pd.to_numeric(frame['timestamp'], errors='coerce').fillna(0).astype('int64')

# Test/validation videos were constrained to the training catalog by preprocessing.
item_ids = sorted(train_df.video_id.unique())
user_ids = sorted(set(train_df.user_id) | set(valid_df.user_id) | set(test_df.user_id))
item2idx = {x:i+1 for i,x in enumerate(item_ids)}
idx2item = {i+1:x for i,x in enumerate(item_ids)}
user2idx = {x:i for i,x in enumerate(user_ids)}
num_items, num_users = len(item2idx), len(user2idx)

def mapped(frame):
    x = frame[frame.video_id.isin(item2idx)].copy()
    x['u'] = x.user_id.map(user2idx).astype('int64')
    x['i'] = x.video_id.map(item2idx).astype('int64')
    return x

train = mapped(train_df); valid = mapped(valid_df); test = mapped(test_df)
train_hist, train_time = {}, {}
for u,g in train.groupby('u', sort=False):
    train_hist[int(u)] = g.i.astype(int).tolist()
    train_time[int(u)] = g.timestamp.astype('int64').tolist()

valid_target = dict(zip(valid.u.astype(int), valid.i.astype(int)))
test_target = dict(zip(test.u.astype(int), test.i.astype(int)))
eval_users = sorted(set(valid_target) & set(test_target) & set(train_hist))

valid_hist = {u:list(train_hist[u]) for u in eval_users}
test_hist = {u:list(train_hist[u])+[valid_target[u]] for u in eval_users}
valid_times = {u:list(train_time[u]) for u in eval_users}
valid_time_lookup = dict(zip(valid.u.astype(int), valid.timestamp.astype('int64')))
test_times = {u:list(train_time[u])+[int(valid_time_lookup[u])] for u in eval_users}

print(f'Users={num_users:,}  Items={num_items:,}  Train interactions={len(train):,}')
print(f'Evaluation users={len(eval_users):,}')
assert set(valid_target[u] for u in eval_users) <= set(range(1,num_items+1))
assert set(test_target[u] for u in eval_users) <= set(range(1,num_items+1))

## 2. Sequential training samples and ranking metrics

Training uses every chronological prefix with at least one preceding video. A negative video is sampled outside the learner's complete training history. Evaluation ranks the target against the complete training catalog and masks already-seen videos.

In [ ]:
def left_pad(values, length, pad=0):
    values = list(values)[-length:]
    return [pad]*(length-len(values)) + values

class PrefixDataset(Dataset):
    def __init__(self, histories, times, max_len, num_items):
        self.histories, self.times = histories, times
        self.max_len, self.num_items = max_len, num_items
        self.examples = [(u,t) for u,seq in histories.items() for t in range(1,len(seq))]
        self.seen = {u:set(seq) for u,seq in histories.items()}
    def __len__(self): return len(self.examples)
    def __getitem__(self, index):
        u,t = self.examples[index]
        seq = left_pad(self.histories[u][:t], self.max_len)
        ts = left_pad(self.times[u][:t], self.max_len)
        pos = self.histories[u][t]
        neg = random.randint(1,self.num_items)
        while neg in self.seen[u]: neg = random.randint(1,self.num_items)
        return (torch.tensor(u), torch.tensor(seq), torch.tensor(ts),
                torch.tensor(pos), torch.tensor(neg))

train_dataset = PrefixDataset(train_hist, train_time, CFG.max_len, num_items)
train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=CFG.num_workers, pin_memory=True, persistent_workers=CFG.num_workers>0)
print('Sequential prefix samples:', len(train_dataset))

def ranking_metrics(ranks, ks=(5,10,20)):
    ranks = np.asarray(ranks)
    out = {'MRR': float(np.mean(1.0/ranks))}
    for k in ks:
        out[f'Recall@{k}'] = float(np.mean(ranks <= k))
        out[f'NDCG@{k}'] = float(np.mean(np.where(ranks <= k, 1/np.log2(ranks+1), 0)))
    return out

@torch.no_grad()
def evaluate_sequential(model, histories, times, targets, users, batch_size=None):
    model.eval(); batch_size = batch_size or CFG.eval_batch_size; ranks=[]
    all_items = torch.arange(1,num_items+1,device=device)
    for start in tqdm(range(0,len(users),batch_size),leave=False,desc='Full-catalog evaluation'):
        us = users[start:start+batch_size]
        seq = torch.tensor([left_pad(histories[u],CFG.max_len) for u in us],device=device)
        ts = torch.tensor([left_pad(times[u],CFG.max_len) for u in us],device=device)
        scores = model.full_scores(seq,ts)  # [B, num_items]
        for row,u in enumerate(us):
            seen = set(histories[u]); target = targets[u]
            seen.discard(target)
            if seen:
                scores[row,torch.tensor([x-1 for x in seen],device=device)] = torch.finfo(scores.dtype).min
            target_score = scores[row,target-1]
            ranks.append(int((scores[row] > target_score).sum().item()+1))
    return ranking_metrics(ranks,CFG.ks)

def bpr_loss(pos,neg): return -F.logsigmoid(pos-neg).mean()

## 3. Baseline 1 — NARM

**Layers:** item embedding → GRU → local/global attention → gated session representation → item scoring. The GRU captures order; attention emphasizes the historical videos most useful for predicting the next video.

In [ ]:
class NARM(nn.Module):
    def __init__(self,n_items,d=128,dropout=.1):
        super().__init__()
        self.n_items=n_items
        self.item_emb=nn.Embedding(n_items+1,d,padding_idx=0)
        self.gru=nn.GRU(d,d,batch_first=True)
        self.q1=nn.Linear(d,d,bias=False); self.q2=nn.Linear(d,d,bias=False)
        self.attn=nn.Linear(d,1,bias=False)
        self.fuse=nn.Linear(2*d,d,bias=False)
        self.drop=nn.Dropout(dropout)
    def encode(self,seq,times=None):
        mask=seq.ne(0); x=self.drop(self.item_emb(seq)); out,_=self.gru(x)
        lengths=mask.sum(1).clamp_min(1); last=out[torch.arange(len(seq),device=seq.device),lengths-1]
        # Histories are left padded, so the actual last state is at column max_len-1.
        last=out[:,-1]
        logits=self.attn(torch.sigmoid(self.q1(out)+self.q2(last).unsqueeze(1))).squeeze(-1)
        # Use the minimum representable value instead of -1e9, which overflows under FP16 AMP.
        logits=logits.masked_fill(~mask,torch.finfo(logits.dtype).min); alpha=torch.softmax(logits,dim=1)
        global_ctx=(alpha.unsqueeze(-1)*out).sum(1)
        return self.fuse(torch.cat([global_ctx,last],dim=-1))
    def pair_scores(self,seq,times,pos,neg):
        h=self.encode(seq,times); return (h*self.item_emb(pos)).sum(-1),(h*self.item_emb(neg)).sum(-1)
    def full_scores(self,seq,times): return self.encode(seq,times) @ self.item_emb.weight[1:].T

## 4. Baseline 2 — SASRec

**Layers:** item and positional embeddings → causal multi-head self-attention blocks → residual normalization → feed-forward network → dot-product ranking. The causal mask prevents a position from seeing future items.

In [ ]:
class SASRec(nn.Module):
    def __init__(self,n_items,max_len=50,d=128,heads=4,layers=2,dropout=.1):
        super().__init__(); self.n_items=n_items; self.max_len=max_len
        self.item_emb=nn.Embedding(n_items+1,d,padding_idx=0); self.pos_emb=nn.Embedding(max_len,d)
        layer=nn.TransformerEncoderLayer(d,heads,4*d,dropout,batch_first=True,norm_first=True,activation='gelu')
        self.encoder=nn.TransformerEncoder(layer,layers); self.norm=nn.LayerNorm(d); self.drop=nn.Dropout(dropout)
    def encode(self,seq,times=None):
        positions=torch.arange(self.max_len,device=seq.device).unsqueeze(0)
        x=self.drop(self.item_emb(seq)+self.pos_emb(positions))
        causal=torch.triu(torch.ones(self.max_len,self.max_len,device=seq.device,dtype=torch.bool),1)
        x=self.encoder(x,mask=causal,src_key_padding_mask=seq.eq(0)); return self.norm(x[:,-1])
    def pair_scores(self,seq,times,pos,neg):
        h=self.encode(seq,times); return (h*self.item_emb(pos)).sum(-1),(h*self.item_emb(neg)).sum(-1)
    def full_scores(self,seq,times): return self.encode(seq,times) @ self.item_emb.weight[1:].T

## 5. Baseline 3 — BERT4Rec

**Layers:** item, position, and mask-token embeddings → bidirectional Transformer blocks → masked-position representation → item scoring. The input is shifted and a mask token is appended, allowing the model to predict the hidden next item without a causal attention mask.

In [ ]:
class BERT4Rec(nn.Module):
    def __init__(self,n_items,max_len=50,d=128,heads=4,layers=2,dropout=.1):
        super().__init__(); self.n_items=n_items; self.max_len=max_len; self.mask_id=n_items+1
        self.item_emb=nn.Embedding(n_items+2,d,padding_idx=0); self.pos_emb=nn.Embedding(max_len,d)
        layer=nn.TransformerEncoderLayer(d,heads,4*d,dropout,batch_first=True,norm_first=True,activation='gelu')
        self.encoder=nn.TransformerEncoder(layer,layers); self.norm=nn.LayerNorm(d); self.drop=nn.Dropout(dropout)
    def masked_input(self,seq):
        # Drop the oldest slot and append [MASK]; works for left-padded and full histories.
        mask_col=torch.full((len(seq),1),self.mask_id,dtype=seq.dtype,device=seq.device)
        return torch.cat([seq[:,1:],mask_col],dim=1)
    def encode(self,seq,times=None):
        xseq=self.masked_input(seq); pos=torch.arange(self.max_len,device=seq.device).unsqueeze(0)
        x=self.drop(self.item_emb(xseq)+self.pos_emb(pos))
        x=self.encoder(x,src_key_padding_mask=xseq.eq(0)); return self.norm(x[:,-1])
    def pair_scores(self,seq,times,pos,neg):
        h=self.encode(seq,times); return (h*self.item_emb(pos)).sum(-1),(h*self.item_emb(neg)).sum(-1)
    def full_scores(self,seq,times): return self.encode(seq,times) @ self.item_emb.weight[1:self.n_items+1].T

## 6. Baseline 4 — LightGCN

**Layers:** trainable user/video embeddings → normalized user–video message propagation for three layers → mean layer aggregation → dot-product ranking. LightGCN intentionally removes nonlinear feature transformations so that the baseline isolates collaborative graph propagation.

In [ ]:
def build_lightgcn_adjacency(train_frame,num_users,num_items):
    u=torch.tensor(train_frame.u.to_numpy(),dtype=torch.long)
    v=torch.tensor(train_frame.i.to_numpy()-1+num_users,dtype=torch.long)
    rows=torch.cat([u,v]); cols=torch.cat([v,u]); n=num_users+num_items
    degree=torch.bincount(rows,minlength=n).float().clamp_min(1)
    values=(degree[rows].rsqrt()*degree[cols].rsqrt())
    return torch.sparse_coo_tensor(torch.stack([rows,cols]),values,(n,n)).coalesce()

class LightGCN(nn.Module):
    def __init__(self,num_users,num_items,adj,d=128,layers=3):
        super().__init__(); self.num_users=num_users; self.num_items=num_items; self.layers=layers
        self.embedding=nn.Embedding(num_users+num_items,d)
        nn.init.normal_(self.embedding.weight,std=.1)
        self.register_buffer('adj',adj)
    def propagate(self):
        x=self.embedding.weight; outputs=[x]
        for _ in range(self.layers): x=torch.sparse.mm(self.adj,x); outputs.append(x)
        z=torch.stack(outputs).mean(0)
        return z[:self.num_users],z[self.num_users:]
    def full_scores(self,users):
        zu,zv=self.propagate(); return zu[users]@zv.T

## 7. Baseline 5 — Heterogeneous Graph Transformer

**Layers:** separate user/video/concept/course embeddings → two HGTConv layers → relation-specific multi-head attention → GELU, residual connection, and layer normalization → user–video dot-product ranking.

The graph uses `watched`, `covers`, and `contains` relations and their reverse directions. This baseline tests whether typed educational relations improve over a collaborative graph alone.

In [ ]:
def build_heterogeneous_graph():
    data=HeteroData()
    data['user'].node_id=torch.arange(num_users)
    data['video'].node_id=torch.arange(num_items)

    # Training watch edges. Dataset item indices start at 1; graph video nodes start at 0.
    uw=torch.tensor(train.u.to_numpy(),dtype=torch.long)
    vw=torch.tensor(train.i.to_numpy()-1,dtype=torch.long)
    data['user','watched','video'].edge_index=torch.stack([uw,vw])
    data['video','rev_watched','user'].edge_index=torch.stack([vw,uw])

    video_lookup={vid:item2idx[vid]-1 for vid in item_ids}
    ccid_lookup={}
    if (GRAPH/'video_index.parquet').exists():
        vm=pd.read_parquet(GRAPH/'video_index.parquet')
        ccid_lookup=dict(zip(vm.video_id.astype(str),vm.ccid.astype(str)))
    ccid_to_video={ccid_lookup[v]:video_lookup[v] for v in video_lookup if v in ccid_lookup and ccid_lookup[v]!='nan'}

    cv=pd.read_parquet(GRAPH/'concept_video_edges.parquet')
    cv['concept_id']=cv.concept_id.astype(str); cv['ccid']=cv.ccid.astype(str)
    cv=cv[cv.ccid.isin(ccid_to_video)].drop_duplicates(['concept_id','ccid'])
    concept_ids=sorted(cv.concept_id.unique()); concept2idx={x:i for i,x in enumerate(concept_ids)}
    data['concept'].node_id=torch.arange(len(concept_ids))
    vc_v=torch.tensor(cv.ccid.map(ccid_to_video).to_numpy(),dtype=torch.long)
    vc_c=torch.tensor(cv.concept_id.map(concept2idx).to_numpy(),dtype=torch.long)
    data['video','covers','concept'].edge_index=torch.stack([vc_v,vc_c])
    data['concept','rev_covers','video'].edge_index=torch.stack([vc_c,vc_v])

    course_count=1
    cp=GRAPH/'course_video_edges.parquet'
    if cp.exists():
        course=pd.read_parquet(cp); course['video_id']=course.video_id.astype(str)
        course=course[course.video_id.isin(video_lookup)].drop_duplicates(['course_id','video_id'])
        course_ids=sorted(course.course_id.astype(str).unique()); course2idx={x:i for i,x in enumerate(course_ids)}
        course_count=max(1,len(course_ids)); data['course'].node_id=torch.arange(course_count)
        if len(course):
            c=torch.tensor(course.course_id.astype(str).map(course2idx).to_numpy(),dtype=torch.long)
            v=torch.tensor(course.video_id.map(video_lookup).to_numpy(),dtype=torch.long)
            data['course','contains','video'].edge_index=torch.stack([c,v])
            data['video','rev_contains','course'].edge_index=torch.stack([v,c])
    if 'course' not in data.node_types: data['course'].node_id=torch.arange(course_count)
    print(data)
    return data

class HGTBaseline(nn.Module):
    def __init__(self,node_counts,metadata,d=128,heads=4,layers=2,dropout=.1):
        super().__init__(); self.drop=dropout
        self.emb=nn.ModuleDict({t:nn.Embedding(node_counts[t],d) for t in node_counts})
        # Current PyG HGTConv aggregates relation messages internally and does not accept `group`.
        self.convs=nn.ModuleList([HGTConv(d,d,metadata,heads=heads) for _ in range(layers)])
        self.norms=nn.ModuleList([nn.ModuleDict({t:nn.LayerNorm(d) for t in node_counts}) for _ in range(layers)])
    def forward(self,data):
        x={t:self.emb[t](data[t].node_id) for t in data.node_types}
        for conv,norm in zip(self.convs,self.norms):
            y=conv(x,data.edge_index_dict)
            x={t:norm[t](x[t]+F.dropout(F.gelu(y.get(t,x[t])),self.drop,self.training)) for t in x}
        return x

## 8. Training functions

Sequential models use mini-batch BPR optimization. Graph models propagate embeddings over the complete filtered graph once per epoch and optimize BPR on all training edges. Validation uses full-catalog NDCG@10 for early stopping.

In [ ]:
def save_checkpoint(model,name,metrics,extra=None):
    torch.save({'model_state':model.state_dict(),'config':asdict(CFG),'metrics':metrics,'extra':extra or {}},
               CHECKPOINTS/f'{name}.pt')

def train_sequential(model,name):
    model=model.to(device); opt=torch.optim.AdamW(model.parameters(),lr=CFG.learning_rate,weight_decay=CFG.weight_decay)
    scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda'); best=-1; bad=0; history=[]
    for epoch in range(1,CFG.epochs+1):
        model.train(); total=0
        for u,seq,ts,pos,neg in tqdm(train_loader,leave=False,desc=f'{name} epoch {epoch}'):
            seq,ts,pos,neg=[x.to(device,non_blocking=True) for x in [seq,ts,pos,neg]]
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type,enabled=device.type=='cuda'):
                ps,ns=model.pair_scores(seq,ts,pos,neg); loss=bpr_loss(ps,ns)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),5.0); scaler.step(opt); scaler.update(); total+=loss.item()*len(seq)
        val=evaluate_sequential(model,valid_hist,valid_times,valid_target,eval_users)
        row={'epoch':epoch,'loss':total/len(train_dataset),**val}; history.append(row); print(name,row)
        if val['NDCG@10']>best:
            best=val['NDCG@10'];bad=0;save_checkpoint(model,name,val)
        else:
            bad+=1
            if bad>=CFG.patience: break
    state=torch.load(CHECKPOINTS/f'{name}.pt',map_location=device);model.load_state_dict(state['model_state'])
    test_metrics=evaluate_sequential(model,test_hist,test_times,test_target,eval_users)
    return model,history,test_metrics

def sample_graph_negatives(users,seen,num_items):
    neg=torch.randint(1,num_items+1,(len(users),),dtype=torch.long)
    for j,u in enumerate(users.tolist()):
        while int(neg[j]) in seen[int(u)]: neg[j]=random.randint(1,num_items)
    return neg

@torch.no_grad()
def evaluate_graph_embeddings(user_z,item_z,histories,targets,users,batch_size=256):
    ranks=[]
    for start in range(0,len(users),batch_size):
        us=users[start:start+batch_size]; ui=torch.tensor(us,device=user_z.device)
        scores=user_z[ui]@item_z.T
        for row,u in enumerate(us):
            seen=set(histories[u]);target=targets[u];seen.discard(target)
            if seen:
                scores[row,torch.tensor([x-1 for x in seen],device=user_z.device)]=torch.finfo(scores.dtype).min
            ranks.append(int((scores[row]>scores[row,target-1]).sum().item()+1))
    return ranking_metrics(ranks,CFG.ks)

def train_lightgcn(model,name='LightGCN'):
    model=model.to(device); opt=torch.optim.Adam(model.parameters(),lr=CFG.graph_learning_rate)
    users=torch.tensor(train.u.to_numpy(),dtype=torch.long,device=device)
    pos=torch.tensor(train.i.to_numpy()-1,dtype=torch.long,device=device); seen={u:set(x) for u,x in train_hist.items()}
    best=-1;bad=0;history=[]
    for epoch in range(1,CFG.epochs+1):
        model.train();opt.zero_grad();zu,zv=model.propagate()
        neg1=sample_graph_negatives(users.cpu(),seen,num_items).to(device)-1
        loss=bpr_loss((zu[users]*zv[pos]).sum(-1),(zu[users]*zv[neg1]).sum(-1))+1e-5*model.embedding.weight.square().mean()
        loss.backward();opt.step()
        model.eval();zu,zv=model.propagate();val=evaluate_graph_embeddings(zu,zv,valid_hist,valid_target,eval_users)
        row={'epoch':epoch,'loss':loss.item(),**val};history.append(row);print(name,row)
        if val['NDCG@10']>best:best=val['NDCG@10'];bad=0;save_checkpoint(model,name,val)
        else:
            bad+=1
            if bad>=CFG.patience:break
    model.load_state_dict(torch.load(CHECKPOINTS/f'{name}.pt',map_location=device)['model_state'])
    model.eval();zu,zv=model.propagate();test_metrics=evaluate_graph_embeddings(zu,zv,test_hist,test_target,eval_users)
    return model,history,test_metrics

def train_hgt(model,data,name='HGT'):
    model=model.to(device);data=data.to(device);opt=torch.optim.AdamW(model.parameters(),lr=CFG.graph_learning_rate,weight_decay=CFG.weight_decay)
    users=torch.tensor(train.u.to_numpy(),dtype=torch.long,device=device)
    pos=torch.tensor(train.i.to_numpy()-1,dtype=torch.long,device=device);seen={u:set(x) for u,x in train_hist.items()}
    best=-1;bad=0;history=[]
    for epoch in range(1,CFG.epochs+1):
        model.train();opt.zero_grad();z=model(data);zu,zv=z['user'],z['video']
        neg=sample_graph_negatives(users.cpu(),seen,num_items).to(device)-1
        loss=bpr_loss((zu[users]*zv[pos]).sum(-1),(zu[users]*zv[neg]).sum(-1))
        loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5);opt.step()
        model.eval()
        with torch.no_grad():z=model(data);val=evaluate_graph_embeddings(z['user'],z['video'],valid_hist,valid_target,eval_users)
        row={'epoch':epoch,'loss':loss.item(),**val};history.append(row);print(name,row)
        if val['NDCG@10']>best:best=val['NDCG@10'];bad=0;save_checkpoint(model,name,val)
        else:
            bad+=1
            if bad>=CFG.patience:break
    model.load_state_dict(torch.load(CHECKPOINTS/f'{name}.pt',map_location=device)['model_state']);model.eval()
    with torch.no_grad():z=model(data);test_metrics=evaluate_graph_embeddings(z['user'],z['video'],test_hist,test_target,eval_users)
    return model,history,test_metrics

## 9. Train selected baselines

Set `RUN_MODELS` to a smaller list while testing. Ten epochs with early stopping are suitable for the first complete experiment. HGT uses the most GPU memory, so run it separately if Colab memory becomes tight.

In [ ]:
RUN_MODELS=['NARM','SASRec','BERT4Rec','LightGCN','HGT']
results=[]; training_histories={}

def record(name,history,test_metrics):
    training_histories[name]=history
    results.append({'Model':name,**test_metrics,'Best_validation_NDCG@10':max(x['NDCG@10'] for x in history)})
    pd.DataFrame(results).to_csv(OUT/'baseline_results_partial.csv',index=False)

if 'NARM' in RUN_MODELS:
    model=NARM(num_items,CFG.hidden_dim,CFG.dropout);model,h,m=train_sequential(model,'NARM');record('NARM',h,m)
    del model;gc.collect();torch.cuda.empty_cache()

if 'SASRec' in RUN_MODELS:
    model=SASRec(num_items,CFG.max_len,CFG.hidden_dim,CFG.heads,CFG.layers,CFG.dropout)
    model,h,m=train_sequential(model,'SASRec');record('SASRec',h,m)
    del model;gc.collect();torch.cuda.empty_cache()

if 'BERT4Rec' in RUN_MODELS:
    model=BERT4Rec(num_items,CFG.max_len,CFG.hidden_dim,CFG.heads,CFG.layers,CFG.dropout)
    model,h,m=train_sequential(model,'BERT4Rec');record('BERT4Rec',h,m)
    del model;gc.collect();torch.cuda.empty_cache()

if 'LightGCN' in RUN_MODELS:
    adjacency=build_lightgcn_adjacency(train,num_users,num_items)
    model=LightGCN(num_users,num_items,adjacency,CFG.hidden_dim,layers=3)
    model,h,m=train_lightgcn(model);record('LightGCN',h,m)
    del model,adjacency;gc.collect();torch.cuda.empty_cache()

if 'HGT' in RUN_MODELS:
    hetero=build_heterogeneous_graph()
    counts={t:int(hetero[t].node_id.numel()) for t in hetero.node_types}
    model=HGTBaseline(counts,hetero.metadata(),CFG.hidden_dim,CFG.heads,CFG.layers,CFG.dropout)
    model,h,m=train_hgt(model,hetero);record('HGT',h,m)
    del model,hetero;gc.collect();torch.cuda.empty_cache()

## 10. Compare and save results

The table uses the untouched chronological test item for every evaluation user. Higher values are better for every reported metric.

In [ ]:
results_df=pd.DataFrame(results).sort_values('NDCG@10',ascending=False).reset_index(drop=True)
display(results_df.style.format({c:'{:.4f}' for c in results_df.columns if c!='Model'}))
results_df.to_csv(OUT/'baseline_test_results.csv',index=False)

with open(OUT/'training_histories.json','w') as f: json.dump(training_histories,f,indent=2)
with open(OUT/'experiment_config.json','w') as f:
    json.dump({**asdict(CFG),'num_users':num_users,'num_items':num_items,
               'train_interactions':len(train),'evaluation_users':len(eval_users),
               'models':RUN_MODELS,'protocol':'full-catalog chronological leave-two-out'},f,indent=2)

ax=results_df.set_index('Model')[[f'NDCG@{k}' for k in CFG.ks]].plot.bar(figsize=(10,5),rot=0,title='Baseline NDCG comparison')
ax.set_ylabel('NDCG');ax.grid(axis='y',alpha=.25)
fig=ax.get_figure();fig.tight_layout();fig.savefig(OUT/'baseline_ndcg_comparison.png',dpi=180,bbox_inches='tight')
print('Saved results and checkpoints to:',OUT)

## Model-layer summary

| Model | Main layers | What it tests |
|---|---|---|
| NARM | Embedding → GRU → local/global attention → fusion → ranking | Recurrent order and session intent |
| SASRec | Item/position embedding → causal Transformer ×2 → ranking | Long-range chronological dependencies |
| BERT4Rec | Item/position/mask embedding → bidirectional Transformer ×2 → masked ranking | Bidirectional contextual sequence learning |
| LightGCN | User/video embeddings → normalized graph propagation ×3 → mean aggregation | Pure collaborative graph signal |
| HGT | Typed embeddings → relation-aware multi-head HGT ×2 → ranking | Educational user–video–concept–course structure |

Use identical chronological splits and full-catalog metrics when comparing these baselines with the proposed CE-THGT model.